In [8]:
# Standard libraries
import sys
# Add your custom path
gems_tco_path = "/Users/joonwonlee/Documents/GEMS_TCO-1/src"
sys.path.append(gems_tco_path)
import os
import logging
import argparse # Argument parsing

# Data manipulation and analysis
import pandas as pd
import numpy as np
import pickle
import torch
import torch.optim as optim
import copy                    # clone tensor
import time

# Custom imports

from GEMS_TCO import kernels_reparam_space_time_gpu as kernels_reparam_space_time

from GEMS_TCO import orderings as _orderings 
from GEMS_TCO import alg_optimization, BaseLogger
from GEMS_TCO import kernels_columns as kernels_reparam_space_time_gpu_col
from typing import Optional, List, Tuple
from pathlib import Path
import typer
import json
from json import JSONEncoder
from GEMS_TCO import configuration as config
from GEMS_TCO.data_loader import load_data2, exact_location_filter
from GEMS_TCO import debiased_whittle as debiased_whittle
from torch.nn import Parameter

Load daily data applying max-min ordering

In [15]:
space: List[str] = ['1', '1']
lat_lon_resolution = [int(s) for s in space]
mm_cond_number: int = 8
years = ['2024']
month_range = [7] 

output_path = input_path = Path(config.mac_estimates_day_path)
data_load_instance = load_data2(config.mac_data_load_path)

#lat_range_input = [1, 3]
#lon_range_input = [125.0, 129.0]

lat_range_input=[-3,2]      
lon_range_input=[121, 131] 
#lat_range_input=[-3,-1]      
#lon_range_input=[121, 125] 

df_map, ord_mm, nns_map, day_offsets = data_load_instance.load_maxmin_ordered_data_bymonthyear(
lat_lon_resolution=lat_lon_resolution, 
mm_cond_number=mm_cond_number,
years_=years, 
months_=month_range,

lat_range=lat_range_input,   
lon_range=lon_range_input,
is_whittle= True
  
)

Loading Strict Whittle Data...
--- Global Monthly Mean for 2024-7: 258.0276 ---


In [16]:
daily_aggregated_tensors_dw = [] 
daily_hourly_maps_dw = []      

daily_aggregated_tensors_vecc = [] 
daily_hourly_maps_vecc = []   

for day_index in range(31):
    hour_start_index = day_index * 8
    hour_end_index = (day_index + 1) * 8
    hour_indices = [hour_start_index, hour_end_index]

    # --- DW용 데이터 로드 (day_offsets 인자 추가) ---
    day_hourly_map, day_aggregated_tensor = data_load_instance.load_working_data(
        df_map, 
        day_offsets,  # <--- 이 부분이 추가되어야 합니다
        hour_indices, 
        ord_mm= None,
        dtype=torch.float64, 
        keep_ori=False
    )
    daily_aggregated_tensors_dw.append(day_aggregated_tensor)
    daily_hourly_maps_dw.append(day_hourly_map)

    # --- Vecchia용 데이터 로드 (day_offsets 인자 추가) ---
    day_hourly_map, day_aggregated_tensor = data_load_instance.load_working_data(
        df_map, 
        day_offsets,  # <--- 이 부분이 추가되어야 합니다
        hour_indices, 
        ord_mm=ord_mm,
        dtype=torch.float64, 
        keep_ori= True
    )
    daily_aggregated_tensors_vecc.append(day_aggregated_tensor)
    daily_hourly_maps_vecc.append(day_hourly_map)

print(f"Aggregated Tensor Shape: {daily_aggregated_tensors_vecc[0].shape}")
# 예상 출력: torch.Size([행수, 12]) -> 열이 12개여야 성공입니다.

Aggregated Tensor Shape: torch.Size([145008, 11])


whittle

In [22]:
import time
import torch
import numpy as np
from torch.nn import Parameter

# Device 설정 (Whittle=CPU)
DEVICE_DW = torch.device("cpu")

# Global Constants
dwl = debiased_whittle.debiased_whittle_likelihood()
TAPERING_FUNC = dwl.cgn_hamming 
NUM_RUNS = 1
DWL_MAX_STEPS = 15      
LAT_COL, LON_COL, VAL_COL, TIME_COL = 0, 1, 2, 3

days_list = [20]  # 0-based index for days (0 to 30 for July)

# --- Main Loop ---
for day_idx in days_list:
    print(f'\n{"="*50}')
    print(f'--- Processing Real Data: Day {day_idx+1} (2024-07-{day_idx+1}) ---')
    print(f'{"="*50}')

    start_time = time.time()

    try:
        # Data Prepare
        daily_hourly_map_dw = daily_hourly_maps_dw[day_idx]
        daily_aggregated_tensor_dw = daily_aggregated_tensors_dw[day_idx].to(DEVICE_DW)

        if daily_aggregated_tensor_dw.shape[0] == 0:
            print(f"Skipping Day {day_idx+1}: No data.")
            continue

        # Initial Parameters (7-Parameter 설정)
        init_sigmasq   = 13.059
        init_range_lat = 0.154 
        init_range_lon = 0.195
        init_advec_lat = 0.0218
        init_range_time = 0.7  
        init_advec_lon = -0.1689
        init_nugget    = 0.247
        
        raw_init_floats = [init_sigmasq, init_range_lat, init_range_lon, init_range_time, 
                            init_advec_lat, init_advec_lon, init_nugget]

        # -------------------------------------------------------
        # STEP 1: Debiased Whittle Pre-process
        # -------------------------------------------------------
        db = debiased_whittle.debiased_whittle_preprocess(
            daily_aggregated_tensors_dw, daily_hourly_maps_dw, day_idx=day_idx, 
            params_list=raw_init_floats, lat_range=[-3, 2 ], lon_range=[121.0, 131.0]
        )

        cur_df = db.generate_spatially_filtered_days(-3, 2, 121, 131).to(DEVICE_DW)
        
        if cur_df.numel() == 0 or cur_df.shape[1] <= max(LAT_COL, LON_COL, VAL_COL, TIME_COL):
            print(f"Error: Data for Day {day_idx+1} is empty or invalid.")
            continue

        unique_times = torch.unique(cur_df[:, TIME_COL])
        time_slices_list = [cur_df[cur_df[:, TIME_COL] == t_val] for t_val in unique_times]

        # -------------------------------------------------------
        # STEP 2: Pre-compute J-vector, Taper Grid
        # -------------------------------------------------------
        print("\nPre-computing J-vector (Hamming taper)...")
        J_vec, n1, n2, p_time, taper_grid = dwl.generate_Jvector_tapered( 
            time_slices_list, tapering_func=TAPERING_FUNC, 
            lat_col=LAT_COL, lon_col=LON_COL, val_col=VAL_COL, device=DEVICE_DW
        )
        
        if J_vec is None or J_vec.numel() == 0 or n1 == 0 or n2 == 0 or p_time == 0:
           print(f"Error: J-vector generation failed for Day {day_idx+1}.")
           continue
           
        print("Pre-computing sample periodogram...")
        I_sample = dwl.calculate_sample_periodogram_vectorized(J_vec)

        print("Pre-computing Hamming taper autocorrelation...")
        taper_autocorr_grid = dwl.calculate_taper_autocorrelation_fft(taper_grid, n1, n2, DEVICE_DW)

        if torch.isnan(I_sample).any() or torch.isinf(I_sample).any():
            print("Error: NaN/Inf in sample periodogram.")
            continue
        if torch.isnan(taper_autocorr_grid).any() or torch.isinf(taper_autocorr_grid).any():
            print("Error: NaN/Inf in taper autocorrelation.")
            continue

        print(f"Data grid: {n1}x{n2}, {p_time} time points. J-vector, Periodogram, Taper Autocorr on {DEVICE_DW}.")

        # -------------------------------------------------------
        # STEP 3: Optimization Loop (NUM_RUNS)
        # -------------------------------------------------------
        all_final_results = []
        all_final_losses = []

        for i in range(NUM_RUNS):
            print(f"\n{'='*30} Initialization Run {i+1}/{NUM_RUNS} {'='*30}")
            
            init_phi2 = 1.0 / init_range_lon
            init_phi1 = init_sigmasq * init_phi2
            init_phi3 = (init_range_lon / init_range_lat)**2
            init_phi4 = (init_range_lon / init_range_time)**2

            initial_params_values = [
                np.log(init_phi1),
                np.log(init_phi2),
                np.log(init_phi3),
                np.log(init_phi4),
                init_advec_lat,
                init_advec_lon,
                np.log(init_nugget)
            ]
            
            print(f"Starting with FIXED params (raw log-scale): {[round(p, 4) for p in initial_params_values]}")

            params_list = [
                Parameter(torch.tensor([val], dtype=torch.float64, device=DEVICE_DW))
                for val in initial_params_values
            ]

            optimizer = torch.optim.LBFGS(
                params_list, lr=1.0, max_iter=DWL_MAX_STEPS, history_size=100, 
                line_search_fn="strong_wolfe", tolerance_grad=1e-5
            )

            print(f"Starting optimization run {i+1} on device {DEVICE_DW} (Hamming, 7-param ST kernel, L-BFGS)...")
            
            # --- 💥 REVISED: Call L-BFGS trainer, pass p_time 💥 ---
            nat_params_str, phi_params_str, raw_params_str, loss, steps_run = dwl.run_lbfgs_tapered(
                params_list=params_list,
                optimizer=optimizer,
                I_sample=I_sample,
                n1=n1, n2=n2, p_time=p_time,
                taper_autocorr_grid=taper_autocorr_grid, 
                max_steps=DWL_MAX_STEPS,
                device=DEVICE_DW
            )
            # --- END REVISION ---
            
            if loss is not None:
                all_final_results.append((nat_params_str, phi_params_str, raw_params_str))
                all_final_losses.append(loss)
            else:
                all_final_losses.append(float('inf'))

        # -------------------------------------------------------
        # STEP 4: Summary Output
        # -------------------------------------------------------
        print(f"\n\n{'='*25} Overall Result from Run {'='*25} {'='*25}")
        
        valid_losses = [l for l in all_final_losses if l is not None and l != float('inf')]

        if not valid_losses:
            print(f"The run failed or resulted in an invalid loss for Day {day_idx+1}.")
        else:
            best_loss = min(valid_losses)
            best_run_index = all_final_losses.index(best_loss)
            best_results = all_final_results[best_run_index]
            
            print(f"Best Run Loss: {best_loss} (after {steps_run} steps)")
            print(f"Final Parameters (Natural Scale): {best_results[0]}")
            print(f"Final Parameters (Phi Scale)    : {best_results[1]}")
            print(f"Final Parameters (Raw Log Scale): {best_results[2]}")

        end_time = time.time()
        print(f"\nTotal execution time: {end_time - start_time:.2f} seconds")

    except Exception as e:
        print(f"🔴 Day {day_idx+1} Failed: {e}")
        import traceback
        traceback.print_exc()
        continue


--- Processing Real Data: Day 21 (2024-07-21) ---

Pre-computing J-vector (Hamming taper)...
Pre-computing sample periodogram...
Pre-computing Hamming taper autocorrelation...
Data grid: 113x158, 8 time points. J-vector, Periodogram, Taper Autocorr on cpu.

============================== Initialization Run 1/1 ==============================
Starting with FIXED params (raw log-scale): [4.2042, 1.6348, 0.4721, -2.5562, 0.0218, -0.1689, -1.3984]
Starting optimization run 1 on device cpu (Hamming, 7-param ST kernel, L-BFGS)...
--- Step 1/15 ---
 Loss: -1.011348 | Max Grad: 8.305031e-02
  Params (Raw Log): log_phi1: 3.4612, log_phi2: 1.8948, log_phi3: 0.1787, log_phi4: -3.7133, advec_lat: -0.0134, advec_lon: -0.2520, log_nugget: -0.0835
--- Step 2/15 ---
 Loss: -1.750051 | Max Grad: 3.603715e-06
  Params (Raw Log): log_phi1: 3.4723, log_phi2: 1.8926, log_phi3: 0.1995, log_phi4: -3.7200, advec_lat: -0.0059, advec_lon: -0.2527, log_nugget: -0.1053
--- Step 3/15 ---
 Loss: -1.750544 | Max Gra